# CropSense Module 1 — MobileNetV2 field adaptation and complete evaluation

**Required architecture:** MobileNetV2 CNN, 38 output classes. **No improvement is claimed before this notebook runs.**

Retain and evaluate the exact existing checkpoint. Fine-tune a copy on PlantDoc **training** photographs mixed with
PlantVillage training images. Select using a held-out portion of PlantDoc training data, then compare on PlantDoc test
and the historical PlantVillage validation set. The field test was inspected in prior work; call it a historical benchmark,
not a newly blinded independent test. PlantVillage validation also previously selected the original checkpoint.

Exact decoded-image duplicates are checked within PlantDoc and between its training/test folders. This does not rule out
near duplicates or the same specimen under different crops. The CNN remains a closed-set classifier: softmax confidence is
not accuracy, and the model still cannot reliably reject unsupported species.

Unlike the historical decision to avoid PlantDoc fine-tuning, this is a **new, versioned adaptation experiment** explicitly
motivated by improving field performance. It does not rewrite the historical domain-shift result.


In [ ]:
PROJECT_MODULE='cropsense-module-1'
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q','scikit-learn==1.6.1','huggingface_hub'],check=True)


In [ ]:
import os, sys, json, hashlib, platform, subprocess, time
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, log_loss)

SEED = 42
np.random.seed(SEED)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = Path('/kaggle/working') / PROJECT_MODULE / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

def write_json(name, value):
    (OUT / name).write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding='utf-8')

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

write_json('environment.json', {'run_id':RUN_ID, 'python':sys.version, 'platform':platform.platform(),
    'packages':{d.metadata['Name']:d.version for d in metadata.distributions()},
    'seed':SEED, 'notebook_status':'executed; results below are generated, not pre-filled'})
results = []

def evaluate(name, split, y_true, probabilities, classes, sample_ids):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    assert probabilities.shape == (len(y_true), len(classes))
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-4)
    pred = probabilities.argmax(axis=1)
    labels = np.arange(len(classes))
    report = classification_report(y_true, pred, labels=labels, target_names=classes,
                                   output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, pred, labels=labels)
    present = np.unique(y_true)
    row = {'model':name, 'split':split, 'n':len(y_true),
           'accuracy':float(accuracy_score(y_true,pred)),
           'balanced_accuracy':float(balanced_accuracy_score(y_true,pred)),
           'macro_f1_all_classes':float(f1_score(y_true,pred,labels=labels,average='macro',zero_division=0)),
           'macro_f1_present_classes':float(f1_score(y_true,pred,labels=present,average='macro',zero_division=0)),
           'weighted_f1':float(f1_score(y_true,pred,average='weighted',zero_division=0)),
           'log_loss':float(log_loss(y_true, probabilities, labels=labels))}
    # Wilson interval is descriptive; correlated field tiles/weeks violate independent-sample assumptions.
    n=len(y_true); p=row['accuracy']; z=1.96
    center=(p+z*z/(2*n))/(1+z*z/n)
    radius=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/(1+z*z/n)
    row['accuracy_wilson95_descriptive']=[float(center-radius),float(center+radius)]
    if classes == ['healthy','moderate_stress','severe_stress']:
        row['healthy_recall']=float(report['healthy']['recall'])
        row['severe_to_healthy_count']=int(cm[2,0])
        row['severe_to_healthy_rate']=float(cm[2,0]/max(cm[2].sum(),1))
    prefix=f'{name}__{split}'
    write_json(prefix+'_metrics.json', {'summary':row,'per_class':report,'class_order':classes,
                                      'confusion_matrix':cm.tolist()})
    pd.DataFrame(report).T.to_csv(OUT/(prefix+'_classification_report.csv'))
    pd.DataFrame(cm,index=classes,columns=classes).to_csv(OUT/(prefix+'_confusion_counts.csv'))
    frame=pd.DataFrame({'sample_id':sample_ids,'true_index':y_true,'pred_index':pred})
    for i,c in enumerate(classes): frame['prob_'+c]=probabilities[:,i]
    frame.to_csv(OUT/(prefix+'_predictions.csv'),index=False)
    fig,axes=plt.subplots(1,2,figsize=(max(12,len(classes)*.65),max(5,len(classes)*.35)))
    norm=np.divide(cm,cm.sum(axis=1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(axis=1,keepdims=True)!=0)
    for ax,data,title in zip(axes,[cm,norm],['Counts','Recall by true class']):
        im=ax.imshow(data,cmap='Blues');fig.colorbar(im,ax=ax,fraction=.04)
        ax.set(xticks=labels,yticks=labels,xticklabels=classes,yticklabels=classes,
               xlabel='Predicted',ylabel='True',title=title)
        ax.tick_params(axis='x',rotation=90)
        if len(classes)<=6:
            for r in labels:
                for c in labels: ax.text(c,r,f'{data[r,c]:.2f}' if data is norm else str(data[r,c]),ha='center',va='center')
    fig.suptitle(prefix);fig.tight_layout();fig.savefig(OUT/(prefix+'_confusion.png'),dpi=150);plt.show();plt.close(fig)
    print(pd.DataFrame(report).T.to_string());print(row)
    results.append(row)
    pd.DataFrame(results).to_csv(OUT/'results_summary.csv',index=False)
    return row

def save_history(name, history):
    pd.DataFrame(history.history).to_csv(OUT/(name+'_history.csv'),index_label='epoch')
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for ax,metric in zip(axes,['loss','accuracy']):
        for key in [metric,'val_'+metric]:
            if key in history.history: ax.plot(np.arange(1,len(history.history[key])+1),history.history[key],label=key)
        ax.set(xlabel='Epoch',title=metric);ax.legend()
    fig.tight_layout();fig.savefig(OUT/(name+'_learning_curves.png'),dpi=150);plt.show();plt.close(fig)

def finish(model_file, contract, decision):
    write_json('model_contract.json', dict(contract, artifact=Path(model_file).name,
                                         sha256=sha256(model_file),run_id=RUN_ID))
    write_json('decision.json',decision)
    pd.DataFrame(results).to_csv(OUT/'results_summary.csv',index=False)
    lines=['# '+PROJECT_MODULE+' — executed experiment report', '', 'Run: '+RUN_ID,
           '', '## Results', '', pd.DataFrame(results).to_csv(index=False),
           '## Decision', '', json.dumps(decision,indent=2),
           '', '## Reproduction', 'See environment.json, config.json, split_manifest.csv, model_contract.json, and this executed notebook.',
           '', '## Limitations', contract.get('limitations','')]
    (OUT/'REPORT.md').write_text('\n'.join(lines),encoding='utf-8')
    import shutil
    archive=shutil.make_archive(str(OUT.parent/(RUN_ID+'_evidence')),'zip',OUT)
    print('Download evidence:',archive)
    print('Candidate export:',model_file,'SHA256:',sha256(model_file))


In [ ]:
CLASSES=['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']
PLANTDOC_MAP={'Apple_Scab_Leaf': 'Apple___Apple_scab', 'Apple_leaf': 'Apple___healthy', 'Apple_rust_leaf': 'Apple___Cedar_apple_rust', 'Bell_pepper_leaf': 'Pepper,_bell___healthy', 'Bell_pepper_leaf_spot': 'Pepper,_bell___Bacterial_spot', 'Blueberry_leaf': 'Blueberry___healthy', 'Cherry_leaf': 'Cherry_(including_sour)___healthy', 'Corn_Gray_leaf_spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_leaf_blight': 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_rust_leaf': 'Corn_(maize)___Common_rust_', 'Peach_leaf': 'Peach___healthy', 'Potato_leaf_early_blight': 'Potato___Early_blight', 'Potato_leaf_late_blight': 'Potato___Late_blight', 'Raspberry_leaf': 'Raspberry___healthy', 'Soyabean_leaf': 'Soybean___healthy', 'Squash_Powdery_mildew_leaf': 'Squash___Powdery_mildew', 'Strawberry_leaf': 'Strawberry___healthy', 'Tomato_Early_blight_leaf': 'Tomato___Early_blight', 'Tomato_Septoria_leaf_spot': 'Tomato___Septoria_leaf_spot', 'Tomato_leaf': 'Tomato___healthy', 'Tomato_leaf_bacterial_spot': 'Tomato___Bacterial_spot', 'Tomato_leaf_late_blight': 'Tomato___Late_blight', 'Tomato_leaf_mosaic_virus': 'Tomato___Tomato_mosaic_virus', 'Tomato_leaf_yellow_virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato_mold_leaf': 'Tomato___Leaf_Mold', 'grape_leaf': 'Grape___healthy', 'grape_leaf_black_rot': 'Grape___Black_rot'}


## 1. Dataset inventory, class mapping, and split provenance
Use the explicitly attached PlantVillage and PlantDoc datasets. Preserve the original alphabetical 38-class order
and the original PlantVillage 85/15 split with seed 42. List unreadable or unmapped field images instead of hiding them.


In [ ]:
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split
tf.keras.utils.set_random_seed(SEED)
print('GPU devices:',tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'Enable a Kaggle GPU before running this training notebook.'

def locate(suffix):
    direct=[Path('/kaggle/input/datasets')/suffix,Path('/kaggle/input')/Path(suffix).relative_to(Path(suffix).parts[0])]
    for p in direct:
        if p.is_dir(): return p
    raise FileNotFoundError(f'Attach the dataset; expected one of {direct}')

pv=locate('abdallahalidev/plantvillage-dataset/color')
pdroot=locate('nirmalsankalana/plantdoc-dataset')
pvtrain=tf.keras.utils.image_dataset_from_directory(pv,validation_split=.15,subset='training',seed=SEED,image_size=(224,224),batch_size=32)
pvval=tf.keras.utils.image_dataset_from_directory(pv,validation_split=.15,subset='validation',seed=SEED,image_size=(224,224),batch_size=32)
assert pvtrain.class_names==CLASSES

def image_frame(paths):
    return pd.DataFrame({'path':list(map(str,paths)),'label':[CLASSES.index(Path(p).parent.name) for p in paths]})

pvtr=image_frame(pvtrain.file_paths);pvv=image_frame(pvval.file_paths)
del pvtrain,pvval
invalid=[]
def field_frame(root):
    records=[]
    for folder in sorted(root.iterdir()):
        if not folder.is_dir(): continue
        if folder.name not in PLANTDOC_MAP:
            invalid.append({'path':str(folder),'reason':'unmapped class directory'});continue
        for path in sorted(folder.iterdir()):
            if not path.is_file(): continue
            try:
                with Image.open(path) as im:
                    im=im.convert('RGB')
                    digest=hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
                records.append({'path':str(path),'label':CLASSES.index(PLANTDOC_MAP[folder.name]),'image_hash':digest})
            except Exception as exc: invalid.append({'path':str(path),'reason':str(exc)})
    return pd.DataFrame(records)

field=field_frame(pdroot/'train');test=field_frame(pdroot/'test')
assert len(field)>100 and len(test)>100
assert field.groupby('image_hash').label.nunique().max()==1,'Conflicting labels on duplicate training images.'
duplicates=field[field.image_hash.isin(test.image_hash)]
duplicates.to_csv(OUT/'excluded_train_test_duplicates.csv',index=False)
field=field[~field.image_hash.isin(test.image_hash)].drop_duplicates('image_hash').reset_index(drop=True)
test=test.drop_duplicates('image_hash').reset_index(drop=True)
assert field.label.value_counts().min()>=2,'Insufficient examples for a stratified field validation split.'
fieldtr,fieldval=train_test_split(field,test_size=.2,random_state=SEED,stratify=field.label)
assert not (set(fieldtr.image_hash)&set(fieldval.image_hash))
pd.DataFrame(invalid,columns=['path','reason']).to_csv(OUT/'unreadable_or_unmapped.csv',index=False)
parts={'pv_train':pvtr,'pv_validation':pvv,'field_train':fieldtr,'field_validation':fieldval,'field_test':test}
pd.concat([p.assign(split=s) for s,p in parts.items()]).to_csv(OUT/'split_manifest.csv',index=False)
pd.concat([p.groupby('label').size().rename(s) for s,p in parts.items()],axis=1).fillna(0).to_csv(OUT/'class_distribution.csv')
print({k:len(v) for k,v in parts.items()});print('Exact train/test field duplicates removed:',len(duplicates))
config={'seed':SEED,'class_order':CLASSES,'plantdoc_mapping':PLANTDOC_MAP,
        'batch_size':32,'steps_per_epoch':200,'head_epochs':2,'fine_tune_epochs':8,'fine_tune_lr':1e-5,
        'field_training_fraction_per_batch':.5,'selection':'field validation macro F1',
        'pv_split':'original 85/15 seed42; historical validation benchmark, not independent test',
        'field_split':'stratified 80/20 inside PlantDoc train; official test reserved for final comparison',
        'pixel_preprocessing':'raw RGB 0-255; x/127.5-1 already inside checkpoint',
        'limitations':'Exact duplicate audit is limited to PlantDoc; near duplicates and cross-dataset image overlap remain possible.'}
write_json('config.json',config)


## 2. Load and verify the original checkpoint
Fetch the existing public model artifact only if it is not attached. Its SHA-256 must match the repository fallback.
Use raw 0–255 RGB values because MobileNetV2 preprocessing is already embedded in this saved graph.


In [ ]:
from huggingface_hub import hf_hub_download
expected='6ea273be92246aa64cd4a6207044fcb8147fabf0ef9e38e07d43ed1adb29f7d4'
local=list(Path('/kaggle/input').rglob('module1_cnn.keras'))
baseline_path=local[0] if local else Path(hf_hub_download('rahulkuntala/cropsense-models','module1_cnn.keras'))
assert sha256(baseline_path)==expected,'Wrong baseline artifact. Do not silently compare different checkpoints.'
baseline=tf.keras.models.load_model(baseline_path,compile=False)
assert baseline.input_shape[1:]==(224,224,3) and baseline.output_shape[-1]==38
write_json('baseline_artifact.json',{'filename':baseline_path.name,'sha256':expected,'input_shape':baseline.input_shape,'output_shape':baseline.output_shape})

def decode(path,label):
    def read(p):
        with Image.open(p.numpy().decode()) as image:
            if image.mode in ('RGBA','LA') or (image.mode=='P' and 'transparency' in image.info):
                rgba=image.convert('RGBA');image=Image.alpha_composite(Image.new('RGBA',rgba.size,'white'),rgba)
            return np.asarray(image.convert('RGB').resize((224,224)),dtype='float32')
    image=tf.py_function(read,[path],tf.float32);image.set_shape((224,224,3))
    return image,label

def dataset(frame,training=False):
    ds=tf.data.Dataset.from_tensor_slices((frame.path.to_numpy(),frame.label.to_numpy(dtype='int32')))
    if training: ds=ds.shuffle(len(frame),seed=SEED,reshuffle_each_iteration=True).repeat()
    return ds.map(decode,num_parallel_calls=tf.data.AUTOTUNE)

valds=dataset(fieldval).batch(32).prefetch(tf.data.AUTOTUNE)
baseline_val=evaluate('original_checkpoint','field_validation',fieldval.label.to_numpy(),baseline.predict(valds,verbose=0),CLASSES,fieldval.path.tolist())


## 3. Controlled adaptation with validation-only selection
Mix field and laboratory training examples so the field domain is represented without entirely forgetting laboratory classes.
Freeze batch-normalization statistics. Save the best field-validation macro-F1 checkpoint across both training phases.
The training ceiling is fixed in advance; the test set is not used for early stopping or checkpoint selection.


In [ ]:
model=tf.keras.models.load_model(baseline_path,compile=False)
backbone=next(layer for layer in model.layers if isinstance(layer,tf.keras.Model) and 'mobilenet' in layer.name.lower())
backbone.trainable=False
augment=tf.keras.Sequential([tf.keras.layers.RandomFlip('horizontal'),tf.keras.layers.RandomRotation(.08),
    tf.keras.layers.RandomZoom(.15),tf.keras.layers.RandomContrast(.2)])
mixed=tf.data.Dataset.sample_from_datasets([dataset(pvtr,True),dataset(fieldtr,True)],weights=[.5,.5],seed=SEED)
mixed=mixed.batch(32).map(lambda x,y:(tf.clip_by_value(augment(x,training=True),0,255),y),num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
checkpoint=OUT/'best_field_validation.keras'

class ValidationSelection(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__();self.best=-1.;self.rows=[]
    def on_epoch_end(self,epoch,logs=None):
        probs=self.model.predict(valds,verbose=0)
        score=f1_score(fieldval.label,probs.argmax(axis=1),labels=np.unique(fieldval.label),average='macro',zero_division=0)
        self.rows.append({'phase':self.phase,'epoch':epoch+1,'field_validation_macro_f1':float(score)})
        pd.DataFrame(self.rows).to_csv(OUT/'checkpoint_selection.csv',index=False)
        if score>self.best:
            self.best=float(score);self.model.save(checkpoint)
        print('Field validation macro F1:',round(score,4),'best:',round(self.best,4))

selection=ValidationSelection();selection.phase='head'
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
history=model.fit(mixed,steps_per_epoch=200,epochs=2,validation_data=valds,callbacks=[selection],verbose=2)
save_history('adapt_head',history)
backbone.trainable=True
for i,layer in enumerate(backbone.layers):
    layer.trainable=i>=100 and not isinstance(layer,tf.keras.layers.BatchNormalization)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5,clipnorm=1.0),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
selection.phase='fine_tune'
history=model.fit(mixed,steps_per_epoch=200,epochs=8,validation_data=valds,callbacks=[selection,
    tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True)],verbose=2)
save_history('adapt_fine_tune',history)
candidate=tf.keras.models.load_model(checkpoint,compile=False)
chosen='adapted_checkpoint' if selection.best>baseline_val['macro_f1_present_classes'] else 'original_checkpoint'
write_json('selection_before_test.json',{'selected':chosen,'original_validation_macro_f1':baseline_val['macro_f1_present_classes'],
                                        'adapted_validation_macro_f1':selection.best})


## 4. Final lab/field evaluation, honest acceptance, and export
Evaluate the original checkpoint and validation-selected checkpoint on the same benchmark images.
Save all 38 columns in the confusion matrix so predictions outside the 27 field classes remain visible.
Report both macro F1 over the full vocabulary and macro F1 over classes actually present in the evaluation set.
Review field gains alongside laboratory regression; a candidate is not promoted automatically.


In [ ]:
evaluated={};selected_model=candidate if chosen=='adapted_checkpoint' else baseline
models_to_test={'original_checkpoint':baseline}
models_to_test[chosen]=selected_model
for name,m in models_to_test.items():
    evaluated[name]={}
    for split,frame in [('field_test',test),('pv_validation',pvv)]:
        ds=dataset(frame).batch(32).prefetch(tf.data.AUTOTUNE)
        evaluated[name][split]=evaluate(name,split,frame.label.to_numpy(),m.predict(ds,verbose=1),CLASSES,frame.path.tolist())
export=OUT/'module1_cnn_candidate.keras';selected_model.save(export)
reloaded=tf.keras.models.load_model(export,compile=False)
fixture=next(iter(valds))[0]
np.testing.assert_allclose(selected_model(fixture,training=False).numpy(),reloaded(fixture,training=False).numpy(),atol=1e-5,rtol=1e-5)
finish(export,{'input_shape':[None,224,224,3],'output_shape':[None,38],'class_order':CLASSES,
    'preprocessing':'raw RGB 0-255 resized to 224x224; transparency composited white; rescaling embedded in graph',
    'limitations':'Closed-set softmax; historically inspected benchmarks; exact duplicate check is not a complete near-duplicate or specimen-level audit.'},
    {'selected':chosen,'reload_predictions_match':True,
     'field_accuracy_gain':evaluated[chosen]['field_test']['accuracy']-evaluated['original_checkpoint']['field_test']['accuracy'],
     'lab_accuracy_change':evaluated[chosen]['pv_validation']['accuracy']-evaluated['original_checkpoint']['pv_validation']['accuracy'],
     'deployment':'candidate only; review lab/field tradeoff, Grad-CAM compatibility and Windows/Mac parity before promotion'})
